# R2 Ablation — Adaptive Gating Without Entropy Regularization

This notebook evaluates the R2 reviewer criticism by comparing:

1. **Fixed RF:NN = 2:1** static fusion (configured as the historical baseline requested for this ablation).
2. **Adaptive gate, $\lambda=0$** — identical gate architecture and tuned hyperparameters, but with entropy regularization disabled.
3. **Proposed adaptive gate, $\lambda=0.05$** — the already-exported trained gates from `models/motor_gate.pt` and `models/total_gate.pt`.

The RF and NN experts, preprocessing, train/validation/test split, gate architecture, optimizer settings, and random seed are kept fixed. The only training change for the no-entropy ablation is setting $\lambda=0$.

> **Important reproducibility note:** the attached IEEE PuneCon paper text describes its stacking base ensemble as a *simple average* before residual correction, not a 2:1 RF:NN weighting. See the note in the final comparison cell. Do not silently label the 2:1 row as the published stacking result unless the 2:1 ratio is documented elsewhere in your source material.

In [1]:
import os
import copy
import json
import random
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Load the fixed data and already-trained experts

This section is copied conceptually from the final dual-gate notebook so that the ablation uses the same preprocessed arrays and the exact exported RF/NN experts.

In [2]:
# Preprocessed data
X_train = np.load("../data/processed/X_train.npy")
X_val = np.load("../data/processed/X_val.npy")
X_test = np.load("../data/processed/X_test.npy")

y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")

y_scaler = joblib.load("../data/processed/y_scaler.pkl")

with open("../models/best_nn_params.json", "r") as f:
    nn_params = json.load(f)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

Shapes:
X_train: (3760, 18)
X_val  : (940, 18)
X_test : (1175, 18)
y_train: (3760, 2)
y_val  : (940, 2)
y_test : (1175, 2)


In [3]:
class FinalParkinsonNN(nn.Module):
    def __init__(self, params, input_dim=18):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, params["units_l1"]), nn.ReLU(),
            nn.Linear(params["units_l1"], params["units_l2"]), nn.ReLU(),
            nn.Dropout(params["dropout"]),
            nn.Linear(params["units_l2"], params["units_l3"]), nn.ReLU(),
            nn.Linear(params["units_l3"], params["units_l4"]), nn.ReLU(),
            nn.Linear(params["units_l4"], 2)
        )

    def forward(self, x):
        return self.net(x)

nn_model = FinalParkinsonNN(nn_params).to(DEVICE)
nn_model.load_state_dict(torch.load("../models/best_nn_model.pt", map_location=DEVICE))
nn_model.eval()

rf_expert = joblib.load("../models/rf_expert.pkl")

with torch.no_grad():
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

    nn_train_preds_scaled = nn_model(X_train_tensor).cpu().numpy()
    nn_val_preds_scaled = nn_model(X_val_tensor).cpu().numpy()
    nn_test_preds_scaled = nn_model(X_test_tensor).cpu().numpy()

rf_train_preds_scaled = rf_expert.predict(X_train)
rf_val_preds_scaled = rf_expert.predict(X_val)
rf_test_preds_scaled = rf_expert.predict(X_test)

print("NN train predictions:", nn_train_preds_scaled.shape)
print("RF train predictions:", rf_train_preds_scaled.shape)

NN train predictions: (3760, 2)
RF train predictions: (3760, 2)


In [4]:
# Gate inputs: original features + RF predictions + NN predictions

gate_train = np.concatenate([X_train, rf_train_preds_scaled, nn_train_preds_scaled], axis=1)
gate_val = np.concatenate([X_val, rf_val_preds_scaled, nn_val_preds_scaled], axis=1)
gate_test = np.concatenate([X_test, rf_test_preds_scaled, nn_test_preds_scaled], axis=1)

print("Gate input shape:", gate_train.shape)

Gate input shape: (3760, 22)


## 2. Recreate the exact gate architecture

The current model uses a two-layer MLP gate (`Linear -> ReLU -> Dropout -> Linear(2)`) followed by temperature-scaled softmax. We reuse this architecture exactly and do **not** retune its hyperparameters for the $\lambda=0$ ablation.

In [5]:
class AdaptiveGate(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate, temperature):
        super().__init__()
        self.temperature = temperature
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        logits = self.net(x)
        return torch.softmax(logits / self.temperature, dim=1)

# Read the exact tuned hyperparameters used by the proposed model.
with open("../models/best_motor_gate_params.json", "r") as f:
    best_motor_gate_params = json.load(f)

with open("../models/best_total_gate_params.json", "r") as f:
    best_total_gate_params = json.load(f)

print("Motor gate params:", best_motor_gate_params)
print("Total gate params:", best_total_gate_params)

Motor gate params: {'hidden_dim': 8, 'learning_rate': 0.0019122748228268711, 'dropout_rate': 0.04470145327296845, 'temperature': 1.0, 'entropy_lambda': 0.05, 'batch_size': 64}
Total gate params: {'hidden_dim': 32, 'learning_rate': 0.0018123769960513428, 'dropout_rate': 0.07826676875662475, 'temperature': 1.0, 'entropy_lambda': 0.05, 'batch_size': 128}


In [6]:
class GateDataset(Dataset):
    def __init__(self, X, y, rf_preds, nn_preds):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.rf_preds = torch.tensor(rf_preds, dtype=torch.float32)
        self.nn_preds = torch.tensor(nn_preds, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.rf_preds[idx], self.nn_preds[idx]


def train_fixed_lambda_gate(
    best_params,
    target_idx,
    entropy_lambda=0.0,
    seed=42,
):
    """Retrain one gate using the already-tuned hyperparameters."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = AdaptiveGate(
        input_dim=gate_train.shape[1],
        hidden_dim=best_params["hidden_dim"],
        dropout_rate=best_params["dropout_rate"],
        temperature=best_params["temperature"],
    ).to(DEVICE)

    optimizer = optim.Adam(
        model.parameters(),
        lr=best_params["learning_rate"],
    )
    mse_loss = nn.MSELoss()

    train_dataset = GateDataset(
        gate_train, y_train, rf_train_preds_scaled, nn_train_preds_scaled
    )
    val_dataset = GateDataset(
        gate_val, y_val, rf_val_preds_scaled, nn_val_preds_scaled
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=best_params["batch_size"],
        shuffle=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=best_params["batch_size"],
        shuffle=False,
    )

    best_val_loss = np.inf
    best_state = None
    patience = 20
    counter = 0
    history = []

    for epoch in range(200):
        model.train()
        train_loss = 0.0
        train_pred_loss = 0.0
        train_entropy = 0.0
        n_train_batches = 0

        for X_batch, y_batch, rf_batch, nn_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            rf_batch = rf_batch.to(DEVICE)
            nn_batch = nn_batch.to(DEVICE)

            optimizer.zero_grad()
            weights = model(X_batch)

            final_pred = (
                weights[:, 0] * rf_batch[:, target_idx]
                + weights[:, 1] * nn_batch[:, target_idx]
            )
            target = y_batch[:, target_idx]

            pred_loss = mse_loss(final_pred, target)
            entropy = -torch.mean(
                torch.sum(weights * torch.log(weights + 1e-8), dim=1)
            )

            loss = pred_loss - entropy_lambda * entropy
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_pred_loss += pred_loss.item()
            train_entropy += entropy.item()
            n_train_batches += 1

        model.eval()
        val_loss = 0.0
        val_pred_loss = 0.0
        val_entropy = 0.0
        n_val_batches = 0

        with torch.no_grad():
            for X_batch, y_batch, rf_batch, nn_batch in val_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                rf_batch = rf_batch.to(DEVICE)
                nn_batch = nn_batch.to(DEVICE)

                weights = model(X_batch)
                final_pred = (
                    weights[:, 0] * rf_batch[:, target_idx]
                    + weights[:, 1] * nn_batch[:, target_idx]
                )
                target = y_batch[:, target_idx]

                pred_loss = mse_loss(final_pred, target)
                entropy = -torch.mean(
                    torch.sum(weights * torch.log(weights + 1e-8), dim=1)
                )
                loss = pred_loss - entropy_lambda * entropy

                val_loss += loss.item()
                val_pred_loss += pred_loss.item()
                val_entropy += entropy.item()
                n_val_batches += 1

        train_loss /= n_train_batches
        train_pred_loss /= n_train_batches
        train_entropy /= n_train_batches
        val_loss /= n_val_batches
        val_pred_loss /= n_val_batches
        val_entropy /= n_val_batches

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_pred_loss": train_pred_loss,
            "train_entropy": train_entropy,
            "val_loss": val_loss,
            "val_pred_loss": val_pred_loss,
            "val_entropy": val_entropy,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history)

## 3. Retrain the no-entropy gates ($\lambda=0$)

Only the entropy coefficient is changed. The tuned hidden dimension, learning rate, dropout, temperature, batch size, split, experts, and early-stopping procedure remain unchanged.

In [7]:
motor_gate_no_entropy, motor_history = train_fixed_lambda_gate(
    best_motor_gate_params,
    target_idx=0,
    entropy_lambda=0.0,
    seed=SEED,
)

print("Motor no-entropy gate trained.")
print("Best validation loss:", motor_history["val_loss"].min())
print("Epochs:", len(motor_history))

Motor no-entropy gate trained.
Best validation loss: 0.011204918225606283
Epochs: 200


In [8]:
total_gate_no_entropy, total_history = train_fixed_lambda_gate(
    best_total_gate_params,
    target_idx=1,
    entropy_lambda=0.0,
    seed=SEED,
)

print("Total no-entropy gate trained.")
print("Best validation loss:", total_history["val_loss"].min())
print("Epochs:", len(total_history))

Total no-entropy gate trained.
Best validation loss: 0.009743488568346947
Epochs: 200


In [9]:
# Persist the ablation gates without overwriting the proposed-model exports.
torch.save(motor_gate_no_entropy.state_dict(), "../models/motor_gate_lambda0.pt")
torch.save(total_gate_no_entropy.state_dict(), "../models/total_gate_lambda0.pt")

motor_history.to_csv("../results/motor_gate_lambda0_training.csv", index=False)
total_history.to_csv("../results/total_gate_lambda0_training.csv", index=False)

print("Saved:")
print("  ../models/motor_gate_lambda0.pt")
print("  ../models/total_gate_lambda0.pt")
print("  ../results/motor_gate_lambda0_training.csv")
print("  ../results/total_gate_lambda0_training.csv")

Saved:
  ../models/motor_gate_lambda0.pt
  ../models/total_gate_lambda0.pt
  ../results/motor_gate_lambda0_training.csv
  ../results/total_gate_lambda0_training.csv


## 4. Load the proposed $\lambda=0.05$ gates

The proposed model was already trained and exported in the original notebook, so we do **not** retrain it here. This avoids unnecessary randomness and preserves the published model state.

In [10]:
def build_gate_from_params(params):
    return AdaptiveGate(
        input_dim=gate_test.shape[1],
        hidden_dim=params["hidden_dim"],
        dropout_rate=params["dropout_rate"],
        temperature=params["temperature"],
    ).to(DEVICE)

motor_gate_proposed = build_gate_from_params(best_motor_gate_params)
total_gate_proposed = build_gate_from_params(best_total_gate_params)

motor_gate_proposed.load_state_dict(
    torch.load("../models/motor_gate.pt", map_location=DEVICE)
)
total_gate_proposed.load_state_dict(
    torch.load("../models/total_gate.pt", map_location=DEVICE)
)

motor_gate_proposed.eval()
total_gate_proposed.eval()

print("Loaded exported lambda=0.05 gates.")

Loaded exported lambda=0.05 gates.


## 5. Same-test-set prediction for all three variants

In [11]:
# Keep the target convention used by the original dual-gate notebook.
# Its exported gate predictions are inverse-transformed with y_scaler and
# evaluated directly against y_test. We preserve that convention here so
# the ablation is directly comparable to the existing manuscript results.
y_test_original = y_test

print("Ground-truth range:", y_test_original.min(axis=0), y_test_original.max(axis=0))

Ground-truth range: [5.1375 7.    ] [37.658 54.992]


In [12]:
def metrics_dict(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Pearson_r": pearsonr(y_true, y_pred)[0],
        "Spearman_rho": spearmanr(y_true, y_pred)[0],
    }

rows = []

for target_name, y_true, preds in [
    ("Motor-UPDRS", y_test_original[:, 0], {
        "Static RF:NN = 2:1": motor_pred_static,
        "Adaptive gate (lambda=0)": motor_pred_l0,
        "Proposed (lambda=0.05)": motor_pred_prop,
    }),
    ("Total-UPDRS", y_test_original[:, 1], {
        "Static RF:NN = 2:1": total_pred_static,
        "Adaptive gate (lambda=0)": total_pred_l0,
        "Proposed (lambda=0.05)": total_pred_prop,
    }),
]:
    for model_name, pred in preds.items():
        m = metrics_dict(y_true, pred)
        rows.append({"Target": target_name, "Model": model_name, **m})

results_df = pd.DataFrame(rows)
results_df

NameError: name 'motor_pred_static' is not defined

## 6. Routing statistics

These help answer the reviewer qualitatively: the no-entropy gate and entropy-regularized gate can be compared not only on prediction error but also on whether the learned routing collapses toward a single expert.

In [ ]:
def routing_stats(weights):
    eps = 1e-8
    entropy = -np.sum(weights * np.log(weights + eps), axis=1)
    return {
        "mean_RF_weight": weights[:, 0].mean(),
        "mean_NN_weight": weights[:, 1].mean(),
        "RF_dominant_pct": 100.0 * np.mean(weights[:, 0] > weights[:, 1]),
        "NN_dominant_pct": 100.0 * np.mean(weights[:, 1] >= weights[:, 0]),
        "mean_entropy": entropy.mean(),
    }

routing_rows = []
for target_name, w0, wp in [
    ("Motor-UPDRS", motor_w_l0, motor_w_prop),
    ("Total-UPDRS", total_w_l0, total_w_prop),
]:
    routing_rows.append({"Target": target_name, "Model": "Adaptive gate (lambda=0)", **routing_stats(w0)})
    routing_rows.append({"Target": target_name, "Model": "Proposed (lambda=0.05)", **routing_stats(wp)})

routing_df = pd.DataFrame(routing_rows)
routing_df

## 7. Paired error comparison

Because every variant is evaluated on the **same test subjects**, the per-sample absolute-error differences are also useful for checking whether the adaptive variants improve over fixed fusion.

In [ ]:
paired_rows = []

for target_name, y_true, static_pred, noent_pred, prop_pred in [
    ("Motor-UPDRS", y_test_original[:, 0], motor_pred_static, motor_pred_l0, motor_pred_prop),
    ("Total-UPDRS", y_test_original[:, 1], total_pred_static, total_pred_l0, total_pred_prop),
]:
    e_static = np.abs(y_true - static_pred)
    e_noent = np.abs(y_true - noent_pred)
    e_prop = np.abs(y_true - prop_pred)

    paired_rows.extend([
        {
            "Target": target_name,
            "Comparison": "lambda=0 vs static 2:1",
            "Mean absolute-error difference (new - baseline)": np.mean(e_noent - e_static),
            "New better (%)": 100.0 * np.mean(e_noent < e_static),
        },
        {
            "Target": target_name,
            "Comparison": "lambda=0.05 vs lambda=0",
            "Mean absolute-error difference (new - baseline)": np.mean(e_prop - e_noent),
            "New better (%)": 100.0 * np.mean(e_prop < e_noent),
        },
        {
            "Target": target_name,
            "Comparison": "lambda=0.05 vs static 2:1",
            "Mean absolute-error difference (new - baseline)": np.mean(e_prop - e_static),
            "New better (%)": 100.0 * np.mean(e_prop < e_static),
        },
    ])

paired_df = pd.DataFrame(paired_rows)
paired_df

## 8. Export ablation results

In [ ]:
os.makedirs("../results", exist_ok=True)

results_df.to_csv("../results/r2_ablation_metrics.csv", index=False)
routing_df.to_csv("../results/r2_ablation_routing.csv", index=False)
paired_df.to_csv("../results/r2_ablation_paired_errors.csv", index=False)

summary = {
    "static_rf_weight": RF_WEIGHT,
    "static_nn_weight": NN_WEIGHT,
    "lambda_no_entropy": 0.0,
    "lambda_proposed": 0.05,
    "motor_gate_no_entropy": "../models/motor_gate_lambda0.pt",
    "total_gate_no_entropy": "../models/total_gate_lambda0.pt",
    "metrics_csv": "../results/r2_ablation_metrics.csv",
    "routing_csv": "../results/r2_ablation_routing.csv",
    "paired_error_csv": "../results/r2_ablation_paired_errors.csv",
}

with open("../results/r2_ablation_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print("Exported R2 ablation artifacts.")

## 9. Important historical-baseline note

The attached IEEE PuneCon paper describes the previous **hybrid stacking ensemble** as follows: predictions from the MLP and RF base models were **combined by simple averaging**, after which a second-level RF learned residual errors. The paper reports a validation MSE of 7.4467 and $R^2=0.9132$ for the hybrid ensemble. It therefore does **not** support calling a fixed 2:1 RF:NN blend the published stacking result.

For the R2 ablation, the 2:1 row above is therefore best treated as a **configured historical/static-fusion baseline** based on the project specification you supplied. If you want the notebook to reproduce the attached paper literally, change:

```python
RF_WEIGHT = 2.0 / 3.0
NN_WEIGHT = 1.0 / 3.0
```

to:

```python
RF_WEIGHT = 0.5
NN_WEIGHT = 0.5
```

That still would not reproduce the paper's *final* residual-correction stack, because that paper's final model includes a second-level RF trained on residuals rather than only the base-model weighted average.